# RadioJOVE — SWNA Pipeline Demo
**Package:** `whitenoise` · **Submodule:** `wn.radiojove`

This notebook walks through Stochastic White Noise Analysis (SWNA) applied to
radio burst observations recorded with a [RadioJOVE](https://radiojove.gsfc.nasa.gov/)
spectrograph. Each cell is one step. Run them in order on a fresh session,
then jump to any individual step to re-run it without re-running everything.

**Framework:** Bernido & Carpio-Bernido (2015), *Methods and Applications of
White Noise Analysis in Interdisciplinary Sciences*

---
## Pipeline overview
```
JPG spectrograph
      │
      │  (RadioJOVE GUI — region selection, Z-score export)
      ▼
RadioJOVE CSV  (time_seconds, intensity_zscore)
      │
      ├──  read_radiojove_csv()    ──►  (time, intensity, metadata)
      │
      ├──  [optional] zscore_normalize() / resample()
      │
      ├──  analyze_burst()         ──►  AnalysisResult + dual MSD plot
      │
      ├──  batch_analyze_bursts()  ──►  DataFrame summary
      │
      └──  compare_bursts()        ──►  ComparisonResult
```

**Model recommendation for radio bursts:**
Use `model='exponential'` for Type III solar bursts and most Jovian S-bursts.
The exponential SWNA model produces a monotonically growing MSD that matches
the fast-rise, slow-decay profile of these events.
The cosine model fails on monotonic MSD data (R² can be as low as −10⁶⁰).

---

## Cell 1 — Imports
Run once per session. Adds the package to the path if not installed,
and fixes Unicode output on Windows.

In [ ]:
import sys, os

# Fix Unicode output on Windows terminals
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Development path — remove this block if whitenoise is installed via pip
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
_REPO_ROOT = os.path.abspath(os.path.join(_NOTEBOOK_DIR, '..', '..', '..'))
_WN_PKG    = os.path.join(_REPO_ROOT, 'whitenoise')
if _WN_PKG not in sys.path:
    sys.path.insert(0, _WN_PKG)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
# Remove the next line if running in an interactive Jupyter environment
matplotlib.use('Agg')

import whitenoise as wn

print('whitenoise version:', wn.__version__)
print('radiojove functions:', [f for f in dir(wn.radiojove) if not f.startswith('_')])

---
## Cell 2 — Configuration
**Edit this block** to point to your RadioJOVE data folder and choose a model.
All downstream cells use these variables.

In [ ]:
# ── Path to your RadioJOVE CSV folder ─────────────────────────────────────────
# Set DATA_FOLDER to a directory of RadioJOVE CSV files.
# If you do not have real data, the next cell creates synthetic CSVs for demo.
DATA_FOLDER = os.path.join(_REPO_ROOT, 'Solar', 'Type 3 Bursts', 'first trials')

# ── SWNA model ─────────────────────────────────────────────────────────────────
# 'exponential'  recommended for Type III solar bursts and Jovian S-bursts
# 'cosine'       for oscillatory MSD (rarely appropriate for radio bursts)
# 'fbm'          fractional Brownian motion (Hurst exponent H)
# wn.list_models() shows all 16 models and their status
MODEL = 'exponential'

# ── Output folder for plots and summary CSV ────────────────────────────────────
OUT_DIR = os.path.join(_REPO_ROOT, 'swna_results_demo')

print(f'DATA_FOLDER : {DATA_FOLDER}')
print(f'Model       : {MODEL}')
print(f'Output dir  : {OUT_DIR}')

---
## Cell 3 — Create synthetic RadioJOVE CSVs (skip if you have real data)

If your `DATA_FOLDER` already contains RadioJOVE CSV files, skip this cell.
Otherwise, this cell generates three synthetic bursts that mimic the
format and characteristics of real Type III solar bursts
(μ ≈ 1.3–1.6, cadence 0.1 s).

**Synthetic data format:**
```
time_seconds,intensity_zscore
0.000,-0.412
0.100, 0.235
...
```
This matches the output of the RadioJOVE region-selection GUI.

In [ ]:
import tempfile

# Check if real data exists; fall back to synthetic
USE_SYNTHETIC = not os.path.isdir(DATA_FOLDER)

if USE_SYNTHETIC:
    print('Real data folder not found — creating synthetic RadioJOVE CSVs...')

    SYNTH_DIR = os.path.join(_NOTEBOOK_DIR, '_synthetic_radiojove')
    os.makedirs(SYNTH_DIR, exist_ok=True)
    DATA_FOLDER = SYNTH_DIR

    np.random.seed(42)
    N       = 400       # 40 seconds at 0.1s cadence
    dt      = 0.1       # seconds
    t_axis  = np.arange(N) * dt

    for region_idx in range(1, 4):
        # Increments: std scales as n^(mu/2 - 0.5) with mu ≈ 1.45
        exponent = 0.45 / 2.0
        increments = np.random.randn(N) * (np.arange(1, N + 1) ** exponent)
        intensity  = np.cumsum(increments)
        # Z-score
        intensity  = (intensity - intensity.mean()) / intensity.std()

        fname = f'230728154000Higgins_Home_region_{region_idx}_0.1s.csv'
        fpath = os.path.join(SYNTH_DIR, fname)
        rows  = np.column_stack([t_axis, intensity])
        header = 'time_seconds,intensity_zscore'
        np.savetxt(fpath, rows, delimiter=',', header=header, comments='', fmt='%.6f')
        print(f'  Created: {fname}')

    print(f'\nSynthetic data folder: {DATA_FOLDER}')
else:
    print(f'Using real data from: {DATA_FOLDER}')

---
## Cell 4 — List and inspect CSV files

In [ ]:
# List all CSVs in the folder
paths = wn.radiojove.list_burst_csvs(DATA_FOLDER)
print(f'{len(paths)} CSV file(s) found:')
for p in paths:
    print(f'  {os.path.basename(p)}')

# Parse metadata from filenames
print('\nFilename metadata:')
for p in paths:
    meta = wn.radiojove.parse_filename_metadata(os.path.basename(p))
    print(f"  station={meta['station']}  region={meta['region']}  cadence={meta['cadence_s']}s")

---
## Cell 5 — Read a single CSV

`read_radiojove_csv()` wraps `wn.read_csv()` and adds filename metadata
(station, region, cadence, datetime) to the returned metadata dict.

In [ ]:
# Read the first file
csv_path = paths[0]
time, intensity, meta = wn.radiojove.read_radiojove_csv(csv_path)

print(f'File     : {os.path.basename(csv_path)}')
print(f'Station  : {meta.get("station", "N/A")}')
print(f'Region   : {meta.get("region", "N/A")}')
print(f'Cadence  : {meta.get("cadence_s", "N/A")} s')
print(f'Datetime : {meta.get("datetime", "N/A")}')
print(f'N points : {len(time)}')
print(f'Duration : {time[-1] - time[0]:.1f} s')
print(f'Intensity range: [{intensity.min():.3f}, {intensity.max():.3f}]')

---
## Cell 6 — Explore the time series

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(time, intensity, lw=0.8, color='#2C3E50', alpha=0.85)
ax.axhline(0, color='#BDC3C7', lw=0.8, linestyle='--')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Intensity (Z-score)', fontsize=11)
ax.set_title(f'RadioJOVE burst — {os.path.basename(csv_path)}', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()

# Save
os.makedirs(OUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUT_DIR, 'burst_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 7 — Analyze a single burst

`analyze_burst()` runs the complete SWNA pipeline and returns a dict
with the AnalysisResult, the dual-panel MSD figure, and a summary row.

**Why `detrend_method='linear'`?**
A linear trend within the burst window is removed before computing the MSD.
This isolates the stochastic fluctuations from any systematic drift
introduced by the burst envelope or receiver baseline.

In [ ]:
out = wn.radiojove.analyze_burst(
    csv_path,
    model=MODEL,
    detrend_method='linear',
    output_dir=OUT_DIR,
)

result = out['result']
if result and result.fit:
    print('\nFit summary:')
    result.summary()
else:
    print('Fitting failed — check the model choice and data quality.')

---
## Cell 8 — Display the MSD plot

The plot was saved automatically by Cell 7 when `output_dir` was set.
Call `plot_burst_msd()` directly to generate a new figure or change the title.

In [ ]:
fig = wn.radiojove.plot_burst_msd(
    out['result'],
    title=f"{meta.get('station', '')} — region {meta.get('region', '')} — {meta.get('cadence_s', '')}s",
)
plt.show()

---
## Cell 9 — (Optional) Preprocessing helpers

The standard RadioJOVE GUI exports Z-scored intensities, so these steps
are usually unnecessary. Use them if your data is in raw ADC counts or
if you need to compare analyses across different cadences.

In [ ]:
# Z-score normalize (only if NOT already done by the GUI pipeline)
# intensity_z = wn.radiojove.zscore_normalize(intensity)

# Downsample from 0.1s to 1.0s cadence
time_1s, intensity_1s = wn.radiojove.resample(time, intensity, target_cadence=1.0)
print(f'Original  : {len(time)} points at {time[1]-time[0]:.1f}s cadence')
print(f'Resampled : {len(time_1s)} points at 1.0s cadence')

# Group all files in the folder by cadence
groups = wn.radiojove.group_by_cadence(paths)
print('\nFiles by cadence:')
for cadence, fpaths in sorted(groups.items(), key=lambda x: (x[0] is None, x[0])):
    print(f'  {cadence}s : {len(fpaths)} file(s)')

---
## Cell 10 — Batch analysis

`batch_analyze_bursts()` processes every CSV in the folder, saves
per-file MSD plots, and returns a summary DataFrame.

Set `n_jobs > 1` for parallel processing (uses `ThreadPoolExecutor`).

In [ ]:
df = wn.radiojove.batch_analyze_bursts(
    DATA_FOLDER,
    model=MODEL,
    detrend_method='linear',
    output_dir=OUT_DIR,
    n_jobs=1,
)

print('\nSummary table:')
if not df.empty:
    display_cols = [c for c in ['dataset', 'mu', 'mu_ci_low', 'mu_ci_high', 'r_squared'] if c in df.columns]
    print(df[display_cols].to_string(index=False))

---
## Cell 11 — Compare bursts at the same cadence

`compare_bursts()` wraps `wn.compare()` with RadioJOVE defaults and
returns a `ComparisonResult`. Use `wn.print_comparison()` for a
quick ASCII table, or `wn.publish_comparison()` for a publication-quality
μ bar chart.

In [ ]:
# Group files by cadence and compare only the 0.1s files
groups = wn.radiojove.group_by_cadence(paths)
cadence_key = sorted([k for k in groups if k is not None])[0]  # smallest cadence

cr = wn.radiojove.compare_bursts(
    groups[cadence_key],
    model=MODEL,
    detrend_method='linear',
)

print(f'\nComparison ({cadence_key}s cadence):')
wn.print_comparison(cr)

# Summary DataFrame is also available
print('\nsummary_df columns:', list(cr.summary_df.columns))

---
## Cell 12 — μ comparison bar chart

In [ ]:
fig = wn.publish_comparison(cr, show=False)
fig.savefig(os.path.join(OUT_DIR, 'comparison_mu.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved: {os.path.join(OUT_DIR, "comparison_mu.png")}')

---
## Notes on using real RadioJOVE data

### Step 1 — Record a spectrum
RadioJOVE receivers produce JPG spectrographs (time on x-axis,
frequency on y-axis, intensity as colour). Typical observing windows
are 5–30 minutes.

### Step 2 — Region selection (GUI)
Use the RadioJOVE region-selection tool to:
1. Load the JPG spectrograph
2. Draw bounding boxes around burst events
3. Export each region as a Z-scored CSV at 0.1s and 1.0s cadence

Output filenames follow the convention:
```
YYMMDDHHMMSS<Station>_region_<N>_<cadence>s.csv
```

### Step 3 — SWNA analysis
Point `DATA_FOLDER` at the folder containing your CSVs and run
this notebook from Cell 4 onwards. No changes to the code are needed.

### Interpreting μ
The package always reports μ (and its 95% CI) directly. For the `exponential`
model used here, the MSD does not reduce to a clean power law, so
classification (`result.regime`) instead comes from the power-law scaling
exponent (μ−1)/2 in the memory function
f(t−τ)h(τ) = (t−τ)^((μ−1)/2)·exp(−β/2τ)/τ:

- μ = 1 → exponent (μ−1)/2 = 0, the power-law term disappears and
  f(t−τ)h(τ) = exp(−β/2τ)/τ — fluctuation behavior is dominated purely by
  exponential damping: **memoryless**.
- μ > 1 → exponent (μ−1)/2 > 0, the memory effect of an earlier fluctuation
  at τ ≪ t is stronger: **non-Markovian, long memory** (past fluctuations
  persist and reinforce).
- μ < 1 → exponent (μ−1)/2 < 0, the memory effect of an earlier fluctuation
  at τ ≪ t is weaker (though it strengthens again as τ approaches t):
  **non-Markovian, short memory**.

Source: Sithi et al. (2025, *Physica Scripta* 100, 015243). (If you
instead fit `cosine`/`sine` to a burst, classification there is on
α = 2μ − 1 — see the main USER_GUIDE / Elnar et al. 2021.) μ = 1 is always
ordinary Brownian motion.

Type III solar bursts typically yield **μ ≈ 1.4–1.6**,
consistent with the fast electron beams that excite them propagating
along open magnetic field lines with persistent, correlated dynamics.

### Reference
Bernido C C, Carpio-Bernido M V (2015).  
*Methods and Applications of White Noise Analysis in Interdisciplinary Sciences.*  
World Scientific.  
ISBN: 978-981-4618-36-8

---